# 열거형

## 열거형의 기본 사용법

- 러스트는 사용자가 지정한 범주값 중 하나만 가질 수 있는 열거형(Enum)을 정의할 수 있음
- `enum` 키워드로 정의
- 열거형이 가질 수 있는 각각의 범주값을 `variant`(변형)라고 부름
- 변형 이름은 관례상 대문자(`SCREAMING_SNAKE_CASE`)로 쓰지 않고 클래스 이름처럼 파스칼케이스(`UpperCamelCase`)를 사용
- 변형을 참조할 때는 구조체 인스턴스의 속성처럼 `.` 기호를 사용하지 않고 `::` 기호를 사용하여 `열거형이름::변형이름` 형태로 접근

```rust
#[derive(Debug)]
enum Shape {
    Triangle,
    Rectangle,
}

fn main() {
    let x: Shape = Shape::Triangle;
    let y: Shape = Shape::Rectangle;
    println!("x={x:?}, y={y:?}");
}
```

- 열거형도 구조체와 마찬가지로 impl 블럭을 사용하여 메서드와 연관함수를 정의할 수 있다.

```rust
#[derive(Debug)]
enum Shape {
    Triangle,
    Rectangle,
}

impl Shape {
    fn lines(&self) -> u16 {
        match self {
            Shape::Triangle => 3,
            Shape::Rectangle => 4,
        }
    }
}

fn main() {
    let x: Shape = Shape::Triangle;
    let y: Shape = Shape::Rectangle;
    println!("x.lines()={}, y.lines()={}", x.lines(), y.lines());
}

```

## 데이터를 가지는 열거형

- 러스트의 열거형은 다른 언어의 열거형과 달리 단순한 범주값 나열에 그치지 않고, 각 variant가 서로 다른 형태의 데이터를 가질 수 있음
  - 이런 특성 때문에 러스트 열거형은 **대수적 자료형(Algebraic Data Type)** 으로 분류되며, 다른 언어의 단순 enum보다 훨씬 강력함
- variant가 가질 수 있는 데이터 형태
  - 데이터 없음: `Quit`
  - 튜플형(이름 없는 필드): `ChangeColor(i32, i32, i32)`
  - 구조체형(이름 있는 필드): `Move { x: i32, y: i32 }`
- 형태가 다른 variant들을 하나의 열거형 타입으로 묶을 수 있다는 점이 핵심

```rust
#[derive(Debug)]
enum Message {
    Quit,                       // 데이터 없음
    Move { x: i32, y: i32 },    // 구조체형 필드
    Write(String),              // 튜플형 필드 1개
    ChangeColor(i32, i32, i32), // 튜플형 필드 3개
}

fn main() {
    let messages = [
        Message::Quit,
        Message::Move { x: 10, y: 20 },
        Message::Write(String::from("hello")),
        Message::ChangeColor(255, 0, 0),
    ];

    for m in &messages {
        println!("{m:?}");
    }
}
```

## match 표현식과 열거형

- 열거형은 `match` 표현식과 함께 사용해 각 variant 별로 분기 처리하는 경우가 많음
- `match`는 모든 variant를 **빠짐없이(exhaustive)** 다뤄야 함
  - 처리하지 않은 variant가 있으면 **컴파일 에러** (런타임 누락 방지)
  - 나머지를 한꺼번에 처리하려면 `_` 또는 변수 패턴 사용
- variant가 가진 데이터는 패턴으로 분해(destructuring)하여 꺼낼 수 있음

```rust
#[derive(Debug)]
enum Shape {
    Triangle,
    Rectangle,
    Circle(f64), // 반지름을 데이터로 가짐
}

fn describe(s: &Shape) -> String {
    match s {
        Shape::Triangle => String::from("삼각형"),
        Shape::Rectangle => String::from("사각형"),
        Shape::Circle(r) => format!("반지름 {r}인 원"), // r 로 데이터 추출
    }
}

fn main() {
    let shapes = [Shape::Triangle, Shape::Rectangle, Shape::Circle(2.5)];
    for s in &shapes {
        println!("{}", describe(s));
    }
}
```

## 표준 라이브러리 열거형: Option 과 Result

- 러스트에는 다른 언어의 `null` 이 없음
  - 값이 있을 수도, 없을 수도 있는 경우를 **`Option<T>`** 열거형으로 명시적으로 표현
  - variant: `Some(T)`(값 있음), `None`(값 없음)
- 실패할 수 있는 연산의 결과는 **`Result<T, E>`** 열거형으로 표현
  - variant: `Ok(T)`(성공값), `Err(E)`(에러값)
- 두 타입 모두 매우 자주 쓰이므로 별도 임포트 없이 `Some`, `None`, `Ok`, `Err` 를 바로 사용 가능
  - 즉, `Option` 과 `Result` 도 결국 데이터를 가지는 열거형의 한 사례

```rust
// 값이 없을 수 있음 -> Option
fn divide(a: f64, b: f64) -> Option<f64> {
    if b == 0.0 {
        None
    } else {
        Some(a / b)
    }
}

// 실패할 수 있음 -> Result
fn parse_number(s: &str) -> Result<i32, std::num::ParseIntError> {
    s.parse::<i32>()
}

fn main() {
    match divide(10.0, 2.0) {
        Some(v) => println!("나눗셈 결과: {v}"),
        None => println!("0으로 나눌 수 없음"),
    }

    match parse_number("123") {
        Ok(n) => println!("파싱 성공: {n}"),
        Err(e) => println!("파싱 실패: {e}"),
    }
}
```

## if let

- 여러 variant 중 **하나의 경우에만** 관심이 있을 때, `match` 대신 `if let` 으로 간결하게 표현 가능
- `if let 패턴 = 값 { ... }` 형태
- `else` 절을 붙여 나머지 경우를 처리할 수도 있음

```rust
fn main() {
    let maybe_value: Option<i32> = Some(5);

    // match 로 쓰면 None 까지 모두 다뤄야 하지만, Some 에만 관심이 있다면:
    if let Some(v) = maybe_value {
        println!("값이 있음: {v}");
    } else {
        println!("값이 없음");
    }
}
```

## 열거형의 메서드

- 구조체와 마찬가지로 `impl` 블록을 사용해 열거형에도 메서드와 연관함수를 정의할 수 있음
- 메서드 내부에서는 보통 `match self` 로 현재 variant 를 판별하여 동작을 분기

```rust
enum Shape {
    Circle(f64),        // 반지름
    Rectangle(f64, f64), // 가로, 세로
}

impl Shape {
    fn area(&self) -> f64 {
        match self {
            Shape::Circle(r) => std::f64::consts::PI * r * r,
            Shape::Rectangle(w, h) => w * h,
        }
    }
}

fn main() {
    let c = Shape::Circle(2.0);
    let r = Shape::Rectangle(3.0, 4.0);
    println!("원 넓이: {:.2}", c.area());
    println!("사각형 넓이: {:.2}", r.area());
}
```